# 🚀 SFT Training — Qwen2.5-0.5B-AITF (dari CPT Lokal)

Fine-tuning Supervised (SFT) dengan **base model hasil CPT** yang disimpan di Google Drive.

> Perbedaan dari notebook sebelumnya: base model **tidak diunduh dari HuggingFace**, melainkan di-load dari folder hasil CPT di Drive.

| # | Langkah | Deskripsi |
|:--|---------|------------|
| 1 | Setup & Install | Dependencies + W&B |
| 2 | Mount Drive | Akses model CPT & dataset |
| 3 | GPU Check | Verifikasi hardware |
| 4 | W&B Login | Setup experiment tracking |
| 5 | Load Dataset | Muat & validasi JSONL |
| 6 | EDA | Eksplorasi token & distribusi |
| 7 | Load Model CPT | Load dari Drive (QLoRA 4-bit) |
| 8 | SFT Training | Train dengan W&B logging |
| 9 | Save & Merge | Simpan adapter & merge |
| 10 | Inferensi | Uji model hasil training |
| 11 | Evaluasi ROUGE | Skor kuantitatif pada test set |
| 12 | Analisis Distribusi Desil | Bandingkan prediksi vs ground truth |

> ⚠️ **Sebelum mulai:** Pastikan folder `Salinan Qwen2.5-0.5B-AITF-CPT_2026-04-06_03-18` dan file `output.jsonl` sudah ada di Google Drive. Aktifkan **Runtime → GPU**.

---
## 1️⃣ Setup & Instalasi Library

In [ ]:
%%capture
!pip install -q \
    "transformers>=4.45.0" \
    "trl>=0.12.0" \
    "peft>=0.13.0" \
    "accelerate>=1.0.0" \
    "bitsandbytes>=0.44.0" \
    "datasets>=3.0.0" \
    "huggingface_hub>=0.25.0" \
    "wandb>=0.18.0" \
    einops

print("✅ Instalasi selesai!")

In [ ]:
import os, json, math, random, warnings, gc, re
from pathlib import Path
from datetime import datetime

import torch
import transformers
import wandb
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

print(f"✅ PyTorch        : {torch.__version__}")
print(f"✅ Transformers   : {transformers.__version__}")
print(f"✅ W&B            : {wandb.__version__}")
print(f"✅ CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU            : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM           : {vram:.1f} GB")

---
## 2️⃣ Mount Google Drive & Konfigurasi Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive ter-mount")

In [ ]:
# ============================================================
#  🔧 SESUAIKAN PATH INI DENGAN LOKASI FILE DI DRIVE ANDA
# ============================================================

DRIVE_ROOT  = "/content/drive/MyDrive/AITF"

# ← Base model hasil CPT (di-load dari Drive, BUKAN dari HuggingFace)
CPT_MODEL_PATH = f"{DRIVE_ROOT}/Salinan Qwen2.5-0.5B-AITF-CPT_2026-04-06_03-18"

# ← Dataset ChatML hasil pipeline ETL
JSONL_PATH  = f"{DRIVE_ROOT}/data_pipeline/data/processed/output.jsonl"

# ← Output SFT
OUTPUT_DIR  = f"{DRIVE_ROOT}/SFT-Qwen2.5-0.5B-AITF/checkpoints"
MERGED_DIR  = f"{DRIVE_ROOT}/SFT-Qwen2.5-0.5B-AITF/merged_model"

# ← (Opsional) HuggingFace Hub
HF_REPO_ID  = ""   # "username/nama-repo"
HF_TOKEN    = ""   # Token dari https://hf.co/settings/tokens

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

print("📁 Konfigurasi Path:")
print(f"   CPT Model  : {CPT_MODEL_PATH}")
print(f"   JSONL      : {JSONL_PATH}")
print(f"   Output     : {OUTPUT_DIR}")
print(f"   Merged     : {MERGED_DIR}")

# Validasi keberadaan file/folder
assert os.path.isdir(CPT_MODEL_PATH), f"❌ Folder CPT tidak ditemukan: {CPT_MODEL_PATH}"
assert os.path.isfile(f"{CPT_MODEL_PATH}/model.safetensors"), "❌ model.safetensors tidak ada di folder CPT!"
assert os.path.isfile(JSONL_PATH), f"❌ File JSONL tidak ditemukan: {JSONL_PATH}"

cpt_size = os.path.getsize(f"{CPT_MODEL_PATH}/model.safetensors") / 1e6
jsonl_size = os.path.getsize(JSONL_PATH) / 1e6
print(f"\n✅ CPT model ditemukan!  (model.safetensors: {cpt_size:.0f} MB)")
print(f"✅ Dataset JSONL ditemukan!  ({jsonl_size:.1f} MB)")

---
## 3️⃣ Verifikasi GPU & Tentukan Compute Config

In [ ]:
!nvidia-smi

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    DTYPE    = torch.bfloat16 if USE_BF16 else torch.float16

    print(f"\n🖥️  GPU    : {gpu_name}")
    print(f"💾 VRAM   : {vram_gb:.1f} GB")
    print(f"🔢 Dtype  : {'bfloat16' if USE_BF16 else 'float16'}")

    if vram_gb >= 40:
        PER_DEVICE_BATCH = 8; GRAD_ACCUM = 2
    elif vram_gb >= 20:
        PER_DEVICE_BATCH = 4; GRAD_ACCUM = 4
    else:  # T4 ~16 GB
        PER_DEVICE_BATCH = 2; GRAD_ACCUM = 8

    EFFECTIVE_BATCH = PER_DEVICE_BATCH * GRAD_ACCUM
    print(f"📦 Batch/device : {PER_DEVICE_BATCH}  (grad_accum={GRAD_ACCUM})")
    print(f"📦 Effective    : {EFFECTIVE_BATCH}")
else:
    print("❌ GPU tidak tersedia — aktifkan Runtime GPU!")
    DTYPE, PER_DEVICE_BATCH, GRAD_ACCUM, USE_BF16 = torch.float32, 1, 1, False

---
## 4️⃣ Weights & Biases — Setup Experiment Tracking

> 🔑 Dapatkan API Key di [wandb.ai/settings](https://wandb.ai/settings) → **API Keys**

In [ ]:
WANDB_API_KEY  = "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"
WANDB_PROJECT  = "qwen2.5-0.5B-Instruct-aitf-cpt-sft"
WANDB_ENTITY   = "redityaimanuel-universitas-br"        
WANDB_RUN_NAME = f"qwen2.5-0.5b-cpt-sft-{datetime.now():%Y%m%d-%H%M}"

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("✅ W&B login berhasil!")
else:
    wandb.login()
    print("✅ W&B login (interaktif)")

os.environ["WANDB_PROJECT"]   = WANDB_PROJECT
os.environ["WANDB_LOG_MODEL"] = "checkpoint"
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY

print(f"\n📊 W&B Config:")
print(f"   Project  : {WANDB_PROJECT}")
print(f"   Run Name : {WANDB_RUN_NAME}")

---
## 5️⃣ Load & Validasi Dataset ChatML

In [ ]:
def load_jsonl(filepath: str) -> list:
    """Load JSONL dan validasi skema system/user/assistant."""
    records, errors = [], 0
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec  = json.loads(line)
                msgs = rec["messages"]
                assert [m["role"] for m in msgs] == ["system", "user", "assistant"]
                assert all(m["content"].strip() for m in msgs)
                records.append(rec)
            except Exception as e:
                errors += 1
                if errors <= 3:
                    print(f"⚠️  Baris {i}: {e}")
    print(f"\n📊 Records valid : {len(records):,}")
    print(f"   Errors/skip    : {errors:,}")
    return records

raw_data = load_jsonl(JSONL_PATH)

print("\n🔍 Contoh record pertama:")
for msg in raw_data[0]["messages"]:
    preview = msg["content"][:120].replace("\n", " ")
    print(f"  [{msg['role'].upper():9s}] {preview}...")

In [ ]:
TRAIN_RATIO = 0.90
VAL_RATIO   = 0.08

# (Opsional) uncomment untuk pakai subset saat debug
# raw_data = raw_data[:5_000]

random.seed(42)
random.shuffle(raw_data)

total   = len(raw_data)
n_train = int(total * TRAIN_RATIO)
n_val   = int(total * VAL_RATIO)

train_data = raw_data[:n_train]
val_data   = raw_data[n_train : n_train + n_val]
test_data  = raw_data[n_train + n_val:]

print(f"📂 Dataset Split:")
print(f"   Train  : {len(train_data):,}  ({TRAIN_RATIO*100:.0f}%)")
print(f"   Val    : {len(val_data):,}   ({VAL_RATIO*100:.0f}%)")
print(f"   Test   : {len(test_data):,}   (sisa)")
print(f"   Total  : {total:,}")

---
## 6️⃣ Eksplorasi Data (EDA)

In [ ]:
sample_n = min(3_000, total)
sample   = random.sample(raw_data, sample_n)

total_toks, user_toks, asst_toks = [], [], []
for rec in sample:
    msgs = rec["messages"]
    total_toks.append(sum(len(m["content"]) // 4 for m in msgs))
    user_toks.append(len(msgs[1]["content"]) // 4)
    asst_toks.append(len(msgs[2]["content"]) // 4)

def stats(lst, label):
    s = sorted(lst); n = len(s)
    print(f"   {label:18s} mean={sum(s)/n:>5.0f}  "
          f"min={s[0]:>4d}  max={s[-1]:>4d}  "
          f"p95={s[int(n*.95)]:>4d}  p99={s[int(n*.99)]:>4d}")

print(f"📊 Token Stats (heuristik, n={sample_n:,}):")
stats(total_toks, "Total/record")
stats(user_toks,  "User content")
stats(asst_toks,  "Assistant")

# Inisialisasi W&B
wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY or None,
    name    = WANDB_RUN_NAME,
    config  = {
        "base_model"     : "CPT-Qwen2.5-0.5B-AITF (lokal)",
        "cpt_model_path" : CPT_MODEL_PATH,
        "method"         : "SFT + QLoRA 4-bit",
        "lora_r"         : 16,
        "lora_alpha"     : 32,
        "dataset"        : "DTSEN-Malang ChatML",
        "total_records"  : total,
        "train_records"  : len(train_data),
        "val_records"    : len(val_data),
        "avg_tokens"     : round(sum(total_toks) / len(total_toks)),
        "max_tokens"     : max(total_toks),
    },
    tags   = ["sft", "qlora", "qwen2.5", "dtsen", "cpt-base", "bahasa-indonesia"],
    reinit = True,
)

# Distribusi desil
print("\n🎯 Distribusi Desil Nasional:")
counter = {}
for rec in sample:
    m = re.search(r"Desil Nasional[:\s]*(\d+)", rec["messages"][2]["content"])
    if m:
        d = int(m.group(1))
        counter[d] = counter.get(d, 0) + 1

wandb.log({"desil_distribution": wandb.plot.bar(
    wandb.Table(columns=["Desil", "Count"],
                data=[[f"Desil {d}", counter[d]] for d in sorted(counter)]),
    "Desil", "Count", title="Distribusi Desil Nasional di Dataset",
)})

for d in sorted(counter):
    bar = "█" * min(counter[d] // 5, 50)
    print(f"  Desil {d:2d}: {counter[d]:4d} {bar}")

print("\n✅ EDA ter-log ke W&B")

---
## 7️⃣ Load Model CPT & Tokenizer (QLoRA 4-bit)

> Model di-load dari **folder CPT di Google Drive** — tidak memerlukan koneksi ke HuggingFace.

In [ ]:
print(f"📥 Loading tokenizer dari CPT lokal: {CPT_MODEL_PATH} ...")
tokenizer = AutoTokenizer.from_pretrained(
    CPT_MODEL_PATH,
    trust_remote_code=True,
    padding_side="right",
    local_files_only=True,   # ← tidak cek HuggingFace Hub
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Vocab size   : {tokenizer.vocab_size:,}")
print(f"✅ pad_token    : {tokenizer.pad_token!r}")
print(f"✅ eos_token    : {tokenizer.eos_token!r}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = DTYPE,
    bnb_4bit_use_double_quant = True,
)

print(f"📥 Loading model CPT (4-bit quantized) dari Drive...")
print(f"   Path: {CPT_MODEL_PATH}")
model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_PATH,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = DTYPE,
    attn_implementation = "eager",
    local_files_only    = True,   # ← tidak cek HuggingFace Hub
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ Model CPT berhasil di-load!")
print(f"   Total params     : {total_params/1e6:.1f}M")
print(f"   Trainable params : {trainable_params/1e6:.1f}M ({trainable_params/total_params*100:.1f}%)")

In [ ]:
lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    inference_mode = False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
wandb.config.update({"trainable_params": trainable}, allow_val_change=True)

---
## 8️⃣ Persiapan Dataset untuk SFTTrainer

In [ ]:
SYSTEM_PROMPT = """Anda adalah sistem ahli analisis profil kesejahteraan sosial yang objektif dan presisi. 

Tugas Anda:
1. Menganalisis kondisi sosial-ekonomi keluarga berdasarkan deskripsi yang diberikan.
2. Memberikan penalaran (reasoning) komprehensif yang mencakup aspek:
- Kepemilikan aset dan kualitas hunian.
- Komposisi anggota keluarga dan beban ketergantungan.
- Stabilitas pendapatan dan akses kebutuhan dasar.

3. Menentukan skor evaluasi internal (0-100) dan estimasi desil nasional (1-10) berdasarkan kriteria kemiskinan makro yang berlaku.

Gunakan format output berikut:
- Analisis Kondisi: (Bedah poin-poin krusial dari deskripsi)
- Reasoning: (Penjelasan mengapa keluarga tersebut masuk ke kategori skor/desil tertentu, hubungkan antar variabel)
- Skor Evaluasi: [Angka]
- Desil Nasional: [Angka 1-10]
"""

def format_chatml(example: dict) -> dict:
    msgs = example["messages"]
    # Override system prompt dengan versi terbaru dari config.yaml
    msgs[0]["content"] = SYSTEM_PROMPT
    text = tokenizer.apply_chat_template(
        msgs,
        tokenize              = False,
        add_generation_prompt = False,
    )
    return {"text": text}

def to_hf_dataset(records: list) -> Dataset:
    raw = Dataset.from_dict({"messages": [r["messages"] for r in records]})
    return raw.map(format_chatml, remove_columns=["messages"])

print("⚙️  Memformat dataset...")
train_ds = to_hf_dataset(train_data)
val_ds   = to_hf_dataset(val_data)
test_ds  = to_hf_dataset(test_data)

print(f"✅ Train  : {len(train_ds):,} samples")
print(f"✅ Val    : {len(val_ds):,} samples")
print(f"✅ Test   : {len(test_ds):,} samples")

print("\n🔍 Preview teks terformat (300 karakter pertama):")
print(train_ds[0]["text"][:300])

---
## 9️⃣ Konfigurasi & Jalankan SFT Training

In [ ]:
MAX_SEQ_LEN = 512
NUM_EPOCHS  = 3
LR          = 2e-4

sft_config = SFTConfig(
    # ── Output ──────────────────────────────────────────────────────────
    output_dir                  = OUTPUT_DIR,
    run_name                    = WANDB_RUN_NAME,

    # ── Epoch & Batch ────────────────────────────────────────────────────
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,

    # ── Optimisasi ───────────────────────────────────────────────────────
    learning_rate               = LR,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    optim                       = "paged_adamw_8bit",
    max_grad_norm               = 1.0,
    fp16                        = not USE_BF16,
    bf16                        = USE_BF16,

    # ── Sequence ─────────────────────────────────────────────────────────
    #max_seq_length              = MAX_SEQ_LEN,
    #dataset_text_field          = "text",
    #packing                     = True,

    # ── Evaluasi & Logging ───────────────────────────────────────────────
    eval_strategy               = "steps",
    eval_steps                  = 200,
    logging_steps               = 50,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    # ── W&B Integration ──────────────────────────────────────────────────
    report_to                   = "wandb",
    logging_first_step          = True,

    # ── Misc ─────────────────────────────────────────────────────────────
    dataloader_num_workers      = 2,
    seed                        = 42,
)

wandb.config.update({
    "num_epochs"     : NUM_EPOCHS,
    "learning_rate"  : LR,
    "max_seq_length" : MAX_SEQ_LEN,
    "effective_batch": PER_DEVICE_BATCH * GRAD_ACCUM,
    "optimizer"      : "paged_adamw_8bit",
    "lr_scheduler"   : "cosine",
    "packing"        : True,
}, allow_val_change=True)

print("✅ SFTConfig siap!")
print(f"   Epochs          : {NUM_EPOCHS}")
print(f"   Effective batch : {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"   Learning rate   : {LR}")
print(f"   Max seq length  : {MAX_SEQ_LEN}")
print(f"   Report to       : wandb ({WANDB_PROJECT})")

In [ ]:
trainer = SFTTrainer(
    model         = model,
    args          = sft_config,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    tokenizer     = tokenizer,
)

print("🚀 Mulai SFT Training...")
print(f"   Base model  : CPT-Qwen2.5-0.5B-AITF (lokal)")
print(f"   Steps total ≈ {len(train_ds) * NUM_EPOCHS // (PER_DEVICE_BATCH * GRAD_ACCUM):,}")
print(f"   W&B Run     : https://wandb.ai/{WANDB_ENTITY or '<username>'}/{WANDB_PROJECT}/runs/{wandb.run.id}")
print()

train_result = trainer.train()

print("\n✅ Training selesai!")
print(f"   Train loss  : {train_result.training_loss:.4f}")
print(f"   Runtime     : {train_result.metrics.get('train_runtime', 0)/60:.1f} menit")
print(f"   Samples/sec : {train_result.metrics.get('train_samples_per_second', 0):.1f}")

---
## 🔟 Simpan Adapter, Merge & Tutup W&B Run

In [ ]:
# ── Simpan LoRA adapter ──────────────────────────────────────────────────
ADAPTER_DIR = OUTPUT_DIR + "/final_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapter tersimpan: {ADAPTER_DIR}")

# Log final metrics ke W&B
wandb.log({
    "final/train_loss"      : train_result.training_loss,
    "final/runtime_min"     : train_result.metrics.get("train_runtime", 0) / 60,
    "final/samples_per_sec" : train_result.metrics.get("train_samples_per_second", 0),
})

In [ ]:
# ── (Opsional) Merge LoRA ke base model CPT ─────────────────────────────
MERGE = True   # Ubah ke False jika ingin skip

if MERGE:
    print("🔀 Merge LoRA adapter ke base model CPT...")
    gc.collect(); torch.cuda.empty_cache()

    # Load ulang base model CPT (full precision untuk merge)
    base_model = AutoModelForCausalLM.from_pretrained(
        CPT_MODEL_PATH,
        torch_dtype      = DTYPE,
        device_map       = "auto",
        trust_remote_code= True,
        local_files_only = True,
    )
    merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    merged = merged.merge_and_unload()

    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print(f"✅ Merged model tersimpan: {MERGED_DIR}")
else:
    print("⏭️  Merge dilewati (MERGE=False)")

In [ ]:
# ── (Opsional) Push ke HuggingFace Hub ──────────────────────────────────
if HF_REPO_ID and HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model_to_push = merged if MERGE else trainer.model
    model_to_push.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"✅ Model dipush ke: https://huggingface.co/{HF_REPO_ID}")
    wandb.log({"hf_model_url": f"https://huggingface.co/{HF_REPO_ID}"})
else:
    print("⏭️  Push ke Hub dilewati")

# ── Tutup W&B run ────────────────────────────────────────────────────────
wandb.finish()
print(f"\n✅ W&B run selesai! Lihat hasil di:")
print(f"   https://wandb.ai/{WANDB_ENTITY or '<username>'}/{WANDB_PROJECT}")

---
## 1️⃣1️⃣ Uji Inferensi Model Hasil SFT

In [ ]:
gc.collect(); torch.cuda.empty_cache()

if MERGE:
    inf_model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True,
    )
    inf_tok = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
else:
    base_m = AutoModelForCausalLM.from_pretrained(
        CPT_MODEL_PATH, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True, local_files_only=True,
    )
    inf_model = PeftModel.from_pretrained(base_m, ADAPTER_DIR)
    inf_tok   = tokenizer

inf_model.eval()
print("✅ Model siap untuk inferensi!")

In [ ]:
def predict(user_text: str, max_new_tokens: int = 150) -> str:
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": user_text},
    ]
    prompt = inf_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = inf_tok(prompt, return_tensors="pt").to(inf_model.device)

    with torch.no_grad():
        output_ids = inf_model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = False,
            repetition_penalty = 1.1,
            pad_token_id       = inf_tok.pad_token_id,
            eos_token_id       = inf_tok.eos_token_id,
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return inf_tok.decode(new_ids, skip_special_tokens=True).strip()

print("=" * 70)
for i, rec in enumerate(test_data[:5], 1):
    user_text  = rec["messages"][1]["content"]
    expected   = rec["messages"][2]["content"]
    prediction = predict(user_text)

    print(f"\n[Sampel {i}]")
    print(f"  Expected   : {expected[:200]}")
    print(f"  Prediction : {prediction[:200]}")
print("=" * 70)

In [ ]:
# ── Uji manual dengan profil buatan ─────────────────────────────────────
user_input = """Profil Keluarga:

[Demografi & Lokasi]
Keluarga ini berlokasi di Kelurahan Mergosono, Kecamatan Kedungkandang,
Kota Malang, Provinsi Jawa Timur. Keluarga terdiri dari 4 orang anggota.
Keluarga ini tidak tercatat sebagai penerima bantuan iuran (non-PBI).

[Kondisi Perumahan]
Mereka menempati rumah berstatus milik sendiri dengan luas lantai 72 meter persegi.
Jenis lantai: keramik; dinding: tembok; atap: genteng.
Sumber air minum utama dari air isi ulang. Penerangan utama menggunakan
listrik PLN dengan meteran dengan daya terpasang 1.300 watt.
Bahan bakar utama untuk memasak adalah gas elpiji 3 kg.

[Kepemilikan Aset]
Aset bergerak yang dimiliki: televisi datar, smartphone, kulkas.
Tidak memiliki lahan atau rumah lain. Tidak memiliki hewan ternak."""

result = predict(user_input)
print(f"\n🤖 Prediksi Model SFT:")
print(result)

---
## 1️⃣2️⃣ Evaluasi: ROUGE & Distribusi Desil pada Test Set

In [ ]:
!pip install -q rouge-score
from rouge_score import rouge_scorer as rs_lib

EVAL_SUBSET = min(500, len(test_data))
eval_sample = test_data[:EVAL_SUBSET]

scorer = rs_lib.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
exact = 0
r1_list, r2_list, rl_list = [], [], []
gt_counter, pred_counter, error_counter = {}, {}, {}

print(f"📊 Evaluasi pada {EVAL_SUBSET:,} sampel test set...")
for i, rec in enumerate(eval_sample):
    user_text = rec["messages"][1]["content"]
    ref       = rec["messages"][2]["content"].strip()
    pred      = predict(user_text).strip()

    if ref.lower() == pred.lower():
        exact += 1

    scores = scorer.score(ref, pred)
    r1_list.append(scores["rouge1"].fmeasure)
    r2_list.append(scores["rouge2"].fmeasure)
    rl_list.append(scores["rougeL"].fmeasure)

    m_gt   = re.search(r"Desil Nasional[:\s]*(\d+)", ref)
    m_pred = re.search(r"Desil Nasional[:\s]*(\d+)", pred)
    if m_gt:
        d = int(m_gt.group(1)); gt_counter[d] = gt_counter.get(d, 0) + 1
    if m_pred:
        d = int(m_pred.group(1)); pred_counter[d] = pred_counter.get(d, 0) + 1
    if m_gt and m_pred:
        err = abs(int(m_gt.group(1)) - int(m_pred.group(1)))
        error_counter[err] = error_counter.get(err, 0) + 1

    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{EVAL_SUBSET}")

em_pct = exact / EVAL_SUBSET * 100
r1_avg = sum(r1_list) / len(r1_list) * 100
r2_avg = sum(r2_list) / len(r2_list) * 100
rl_avg = sum(rl_list) / len(rl_list) * 100

print(f"\n📊 Hasil Evaluasi (n={EVAL_SUBSET:,}):")
print(f"   Exact Match : {em_pct:>6.2f}%")
print(f"   ROUGE-1     : {r1_avg:>6.2f}%")
print(f"   ROUGE-2     : {r2_avg:>6.2f}%")
print(f"   ROUGE-L     : {rl_avg:>6.2f}%")

# Distribusi error desil
total_parsed = sum(error_counter.values())
if total_parsed > 0:
    print("\n📊 Distribusi Error Desil (|prediksi - truth|):")
    for err in sorted(error_counter):
        pct = error_counter[err] / total_parsed * 100
        bar = "█" * min(int(pct), 40)
        print(f"  Selisih {err:>2}: {error_counter[err]:>4} ({pct:>5.1f}%) {bar}")
    exact_desil_pct = error_counter.get(0, 0) / total_parsed * 100
    within1_pct     = (error_counter.get(0,0) + error_counter.get(1,0)) / total_parsed * 100
    print(f"\n   Exact desil     : {exact_desil_pct:.1f}%")
    print(f"   Within ±1 desil : {within1_pct:.1f}%")

# Log ke W&B
wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY or None,
    name=WANDB_RUN_NAME + "-eval", reinit=True,
)
wandb.log({
    "eval/exact_match_pct" : em_pct,
    "eval/rouge1"          : r1_avg,
    "eval/rouge2"          : r2_avg,
    "eval/rougeL"          : rl_avg,
    "eval/n_samples"       : EVAL_SUBSET,
    "eval/exact_desil_pct" : exact_desil_pct if total_parsed > 0 else 0,
    "eval/within1_desil_pct": within1_pct if total_parsed > 0 else 0,
})
wandb.finish()
print("\n✅ Hasil evaluasi ter-log ke W&B!")

---
## 📊 Ringkasan

| Item | Nilai |
|------|-------|
| Base Model | CPT-Qwen2.5-0.5B-AITF (lokal dari Drive) |
| Metode | SFT + QLoRA 4-bit (NF4) |
| LoRA Rank | 16 (alpha=32) |
| Dataset | DTSEN ChatML JSONL |
| Epochs | 3 |
| Optimizer | paged_adamw_8bit |
| Max Seq Len | 512 token |
| Experiment Tracking | Weights & Biases |

### 🔗 File yang Dihasilkan
- `SFT-Qwen2.5-0.5B-AITF/checkpoints/final_adapter/` — LoRA adapter
- `SFT-Qwen2.5-0.5B-AITF/merged_model/` — Model SFT final setelah merge dengan base CPT